## 1. Motivation and model overview

The **ball-drive risk** component estimates the probability that the current ball carrier loses possession during a canonical one-second ball drive. It is the action-specific risk $d_t$ used by PRESS to construct the possession-loss hazard. The event concerns loss of the current player's control during the drive, not a failed pass.

The model combines two complementary research traditions. Bekkers' pressing-intensity formulation motivates estimating how quickly defenders can reach a moving ball carrier [@bekkers2025]. Spearman's pitch-control framework motivates representing arrival uncertainty and the rate at which a player can establish control [@spearman2017; @spearman2018beyond]. PRESS adapts these ideas to a player **already in possession**: a defender must be able to reach the carrier and, once reachable, can generate a successful defensive intervention at a certain *tackle rate*. This rate plays a role analogous to the control rate in pitch control, but measures the intensity of possession loss. PRESS does not reproduce Bekkers' aggregate pressing score.

Each defender contributes a time-varying cause-specific hazard. A shared survival process combines these contributions into the probability of retaining the ball over one second. The one-second action convention follows the modular EPV perspective of Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021].

The approach is **physics-informed**: motion and reachability follow an explicit physical approximation, whereas tackle intensity is estimated from observed outcomes. We first introduce a simple log-linear, proportional-hazards-inspired rate and then extend it with penalized additive spline effects. The final fitted spline model is used for inference.

## 2. Motion and defender interception time

### Projected ball-carrier trajectory

Let $x_B(0)$ and $v_B(0)$ denote the carrier's observed position and velocity at the evaluation frame. During the hypothetical drive, the carrier follows a constant-velocity projection:

$$
x_B(t)=x_B(0)+v_B(0)t,\qquad 0\le t\le 1.
$$

At each integration time, defender $i$ targets the projected carrier position $x_B(t)$. The carrier is represented as a 2d point. Let $d_i(t)$ denote the defender's distance to that target, and let $u_i=\lVert v_i(0)\rVert$ be the defender's observed initial speed.

### Travel and turning time

For initial speed $u_i\le v_{\max}$, the motion approximation uses linear acceleration $a=a_{\mathrm{linear}}$ up to an effective speed $v_{\max}$. Define

$$
t_a=\max\left\{\frac{v_{\max}-u_i}{a},0\right\},
\qquad
d_a=u_it_a+\frac12 at_a^2.
$$

The estimated travel time to the projected target is

$$
t_{\mathrm{move},i}(t)=
\begin{cases}
\dfrac{\sqrt{u_i^2+2ad_i(t)}-u_i}{a}, & d_i(t)\le d_a,\\[6pt]
t_a+\dfrac{d_i(t)-d_a}{v_{\max}}, & d_i(t)>d_a.
\end{cases}
$$

If the observed initial speed exceeds $v_{\max}$, the model retains that initial speed without additional acceleration and uses $t_{\mathrm{move},i}(t)=d_i(t)/u_i$. Thus $v_{\max}$ is not a hard cap on observed initial speeds.

Following the geometric approximation introduced by Pleuler [@pleuler2020handbook], a simplified turning delay accounts for the defender's need to realign with the target. With turning angle $\theta_i(t)$ in radians and turning-acceleration parameter $a_{\mathrm{turn}}$,

$$
t_{\mathrm{turn},i}(t)=\frac{u_i\theta_i(t)}{a_{\mathrm{turn}}}.
$$

This expression follows from a turning radius $R=u_i^2/a_{\mathrm{turn}}$ and arc time $R\theta_i/u_i$. The implementation adds a delay but does not propagate the defender along a turning arc. The arrival-time estimate is

$$
\tau_i(t)=t_{\mathrm{move},i}(t)+t_{\mathrm{turn},i}(t).
$$

For the existing near-target convention, $\tau_i(t)=0$ whenever $d_i(t)\le0.1$ m. This does not imply certain reachability.

## 3. Probabilistic defender reachability

As in the pitch-control approach, deterministic arrival times are converted into a smooth reachability gate. With logistic arrival-time dispersion $s$ (in seconds), defender $i$ has reachability

$$
F_i(t)=\left[1+\exp\left\{-\frac{\pi\,[t-\tau_i(t)]}{\sqrt{3}\,s}\right\}\right]^{-1}.
$$

The parameter $s$ is the standard deviation of the corresponding logistic arrival-time uncertainty. The gate increases when the defender can reach the projected carrier position sooner relative to elapsed time.

## 4. Tackle intensity

### From a log-linear rate to additive splines

The second ingredient is the instantaneous rate at which a reachable defender can cause loss of control. Inspired by the control-rate mechanism of pitch control, PRESS assigns defender $i$ a nonnegative tackle rate $\lambda_i(t)$, with units s$^{-1}$. Unlike a probability, a rate may exceed one.

A simple starting point is a log-linear, proportional-hazards-inspired model:

$$
\lambda_i(t)=\lambda_0\exp\!\left\{\beta_v\lVert v_B\rVert+\beta_B B(t)\right\}.
$$

Here $\lambda_0$ is a baseline rate, $\lVert v_B\rVert$ is carrier speed, and $B(t)$ is signed minimum distance to the pitch boundaries. This model assumes linear covariate effects on the log-rate. The final PRESS specification relaxes that restriction through penalized additive spline effects and adds a defender-specific approach angle:

$$
\boxed{\log\lambda_i(t)=\alpha+g_v(\lVert v_B\rVert)+g_B(B(t))+g_\phi(\phi_i(t)).}
$$

The fitted specification uses linear terms plus restricted cubic nonlinear terms for speed and boundary distance; angle has a standardized linear term and two nonlinear terms. There are no interaction terms. The model retains a multiplicative rate structure, but its full cause-specific hazards are time-varying through reachability and the evaluated covariates. It should not be interpreted as a textbook Cox model with constant hazard ratios.

### Why these covariates?

**Carrier speed.** Schepers et al. identify movement speed as an important predictor of dribbling outcomes [@schepers2025dribble]. Greater speed can make fine ball control more difficult, increasing vulnerability to intervention. The fitted PRESS model supports an increase in tackle intensity at higher carrier speeds. The spline allows the magnitude and shape of that effect to be estimated without imposing a strictly log-linear response.

**Boundary proximity.** Available space is a central aspect of defensive pressure, including SkillCorner's emphasis on spatial constraints [@skillcorner2026]. Near a boundary, the carrier has fewer possible directions in which to evade an intervention. PRESS uses the signed minimum distance to **all four** pitch boundaries, recomputed along the projected path:

$$
B(t)=\min\!\left\{\frac L2-x_B(t),\frac L2+x_B(t),\frac W2-y_B(t),\frac W2+y_B(t)\right\}.
$$

This differs from a sideline-only distance and can become negative beyond the pitch boundary.

**Defender approach angle.** Spatial constraints also depend on whether a defender approaches from ahead, behind, or the side. Motivated by the role of the carrier's available field of view and escape directions in pressure interpretation, PRESS measures $\phi_i(t)$ as the angle between carrier velocity and the defender's predicted final approach direction towards $x_B(t)$. It is a geometric proxy, not observed body orientation or gaze. The angle contribution is masked when carrier speed is below 1 m/s or the direction is undefined.

These three covariates describe complementary mechanisms: control difficulty at speed, geometric restrictions near the pitch boundary, and the direction of defensive engagement. The spline terms quantify their associations in the fitted model.

## 5. From defender hazards to possession-loss probability

Defender $i$ contributes the effective cause-specific hazard

$$
h_i(t)=F_i(t)\lambda_i(t).
$$

Reachability gates the tackle intensity: even a high underlying tackle rate contributes little when the defender is unlikely to reach the carrier. Under the model's competing-risk construction, all defenders share a single survival probability,

$$
S(t)=\exp\!\left\{-\int_0^t\sum_i h_i(u)\,du\right\}.
$$

The probability that defender $i$ is the cause of loss by time $t$ is

$$
P_i(t)=\int_0^t S(u)h_i(u)\,du.
$$

Therefore,

$$
\boxed{P(\mathrm{loss\ by}\ t\mid\mathrm{drive})=1-S(t)=\sum_i P_i(t).}
$$

The **one-second ball-drive risk** passed to the wider PRESS framework is $d_t=1-S(1)$, where the evaluation state is understood to be fixed at the start of the hypothetical action. Multiple defenders affect risk through their individual reachability and rates; coordinated defensive behaviour is not explicitly modelled.

### Numerical integration

The implementation uses exponential Euler integration. Over a step of width $\Delta$, hold the hazards fixed and write $H=\sum_i h_i$. The step loss mass is $S(1-e^{-H\Delta})$ and, for $H>0$, defender $i$ receives the fraction $h_i/H$. Survival is updated as

$$
S\longleftarrow Se^{-H\Delta}.
$$

This allocation preserves probability mass across the competing causes.

## 6. Estimation and fitted parameters

### Training outcome and cohort

Each canonical one-second section contributes a Bernoulli outcome. A loss before the end of the section has likelihood contribution $1-S(1)$; a completed successful section contributes $S(1)$. The target is individual ball-control loss during a drive.

The original training cohort excludes successful short tails, goalkeeper carriers, and individual possessions ending in a shot. Starts and ends of individual possessions must be matched. Loss-ending possessions are retained only when associated with a matched on-ball engagement (OBE); OBE is a **cohort criterion, not a predictor**, and is not required for successful-ending possessions. Removing 20 unverified loss parents removes 34 sections, including 14 preceding successful sections. The resulting cohort comprises **9,114 sections, of which 456 are losses (5.003%)**.

Provider-labelled individual control loss may include a ball going out of play even when the same team retains the restart. Additional tracking snapshots may be evaluated at inference time, but they do not create new action-selection training observations. The final model uses the natural outcome proportions rather than loss-class resampling and estimates average rather than player-specific skill.

### Motion parameters

The default motion settings are $v_{\max}=7$ m/s and $a_{\mathrm{linear}}=a_{\mathrm{turn}}=7$ m/s$^2$. They are informed by an audit of matched defensive engagements:

| Defensive engager quantity | Conditioning | N selected | P95 | P99 |
|---|---|---:|---:|---:|
| Speed in event frames (m/s) | Speed $>1$ | 95,236 | 6.954 | 8.002 |
| Peak speed per engagement (m/s) | Both carrier and engager peaks $>1$ | 6,197 | 7.248 | 8.236 |
| Peak acceleration magnitude per engagement (m/s$^2$) | Peak magnitude $>3$ | 3,877 | 6.000 | 6.966 |

The acceleration sample comes from 6,677 matched engagements before filtering. Event-frame observations can be repeated when engagements overlap. These are **conditional empirical quantiles**, not physiological ceilings. Seven m/s represents an effective high engagement speed, while seven m/s$^2$ is close to the conditional P99 acceleration magnitude and matches the acceleration used by Spearman [@spearman2018beyond]. Spearman's reference sustainable speed is 5 m/s, not the 7 m/s adopted here. Smoothed tracking acceleration does not independently identify maximal tangential and centripetal acceleration; equating the two acceleration parameters is a modelling convention, and both remain configurable.

The runtime uses positions and velocities rather than observed accelerations. Its seven-frame centred velocity estimate makes the present evaluation offline rather than strictly causal online. The [engagement quantiles](../../data/benchmarks/ball_drive/engagement_quantiles.csv) and [provenance record](../../data/benchmarks/ball_drive/provenance.json) document the source and filters.

### Arrival dispersion and parameter uncertainty

The final ten-match refit estimates $s=0.299632$ s and $\lambda_0=0.344563$ s$^{-1}$. The latter depends on the fitted basis convention and should not be interpreted as an average possession-loss probability.

A joint maximum-a-posteriori fit and local Laplace approximation provide an approximate marginal distribution for $\ell=\log s$:

$$
\ell\mid D\ \dot\sim\ \mathcal N(-1.205201,\,0.003378886).
$$

| Quantity | Final ball-drive refit |
|---|---:|
| Fitted $s=\exp(\hat\ell)$ | 0.299632 s |
| Approximate posterior SD of $s$ (delta method) | 0.017417 s |
| 68% credible interval | 0.282802–0.317463 s |
| 95% credible interval | 0.267368–0.335789 s |

The equal-tailed interval at coverage $c$ is obtained by transforming $\hat\ell\pm z_{(1+c)/2}\sqrt{\Sigma_{\ell\ell}}$ back to seconds. The covariance is the marginal of the **full joint fitted covariance**, including the tackle coefficients and priors or penalties. These summaries describe parameter uncertainty conditional on the selected model, cohort, tracking preparation, and penalty strengths. They do not account for model-selection uncertainty, systematic annotation errors, or within-match dependence, and they are not out-of-fold performance estimates. See the [machine-readable Laplace results](../../data/benchmarks/ball_drive/arrival_dispersion_laplace.json).

The dispersion is model-specific. Spearman's pass model reports 0.45 s, and a later OBSO formulation reports a MAP estimate of 0.54 s under a different estimation setup [@spearman2017; @spearman2018beyond]. Differences in movement approximation, data preparation, cohort selection, and tackle-rate flexibility may affect the fitted $s$, but their separate effects have not been identified here. A smaller estimate does **not** establish better tracking quality or less physiological variability. The [pass-probability vignette](../02_pass_probability/02_pass_probability.ipynb) discusses its independently fitted arrival parameter.

## 7. Evaluation and interpretation

The final physical spline model achieves pooled out-of-fold cross-entropy **0.143483**, a **27.80%** improvement relative to the training-prevalence null, and ten-quantile expected calibration error **0.475%**. Each evaluation row is predicted by a model trained without its match; these are not the in-sample scores of the final full-data export.

A separate native CatBoost experiment yields cross-entropy **0.139855** with natural training and **0.139928** after training-only 10% loss undersampling with analytical prior correction. The predictive gain is modest and does not replace the physics-informed model in PRESS. The [dedicated benchmark vignette](../../benchmarks/01_ball_drive_risk.qmd) contains the grouped evaluation, calibration, comparisons with published approaches, and the log-linear versus spline analysis. Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021] and Stalpaert et al. [@stalpaert2022] provide the main task-level points of reference; differences in data and evaluation protocols must be considered.

### Scope and limitations

The model uses a point-carrier approximation, constant carrier velocity over the hypothetical drive, simplified defender turning, and a reachability-gated tackle process. It does not explicitly represent body posture, foot-level ball control, individual skill, or fully coordinated defensive behaviour. Schepers et al. [@schepers2025dribble] motivate investigating richer dribbling and posture features, but overlap between successful and unsuccessful examples in the current 2D covariates does not establish that all useful 2D information has been exhausted. Shorter-horizon probabilities follow from the survival formulation but have not been separately calibrated.

## 8. Application to tracking data

For a selected frame in a real SkillCorner match, the fitted model takes the carrier's position and velocity together with the defenders' positions and velocities. It returns the predicted loss probability over the requested horizon. A pitch visualization can show the carrier and defenders in their observed configuration, while the cause-specific contributions and survival curve explain how that configuration produces risk.

The application should use the real-match loading workflow described in the [match guide](../../usage.qmd) and [Python API](../../api.qmd), with `BallDriveModel.pretrained()` supplied as the fitted component. The earlier notebook's `load_demo_match()` example and synthetic speed–boundary grid are intentionally **not** presented here as real-match evidence. They should be replaced with a verified SkillCorner frame and its computed outputs before publishing a fully executable real-match example.

## References and next steps

Bekkers [@bekkers2025] motivates moving-target pressing and reachability. Spearman [@spearman2017; @spearman2018beyond] motivates arrival uncertainty and control rates. Fernández, Bornn and Cervone [@fernandez2019; @fernandez2021] motivate the canonical action decomposition. Stalpaert et al. [@stalpaert2022] provide a carry-model benchmark, and Schepers et al. [@schepers2025dribble] motivate the dribbling covariates. SkillCorner [@skillcorner2026] motivates the interpretation of spatial constraints.

The equations above describe the PRESS formulation rather than a reproduction of any single cited model. The legacy `integrate_loss_prob` interface and `PRESS` without an explicitly fitted component retain the historical log-linear behaviour. The pass-success component is described in the [next model vignette](../02_pass_probability/02_pass_probability.ipynb).
